<a href="https://colab.research.google.com/github/Progammingfrenzy/diabetes-ml-classification/blob/main/Diabetes_FT_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import torch
import sklearn

print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Scikit-learn", sklearn.__version__)

Pandas: 2.2.3
PyTorch: 2.11.0+cpu
Scikit-learn 1.6.1


In [5]:
import os

#Show files currently uploaded to Colab
print(os.listdir("/content"))

['.config', 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv', 'diabetes_binary_health_indicators_BRFSS2015.csv', '.ipynb_checkpoints', 'diabetes_012_health_indicators_BRFSS2015.csv', 'sample_data']


In [6]:
# Load the three-class diabetes dataset
df = pd.read_csv("/content/diabetes_012_health_indicators_BRFSS2015.csv")

# Display dataset dimensions
print("Dataset shape", df.shape)

# Show the first 5 rows
display(df.head())

# Check how many samples belong to each class
print(df["Diabetes_012"].value_counts().sort_index())

Dataset shape (253680, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


Diabetes_012
0.0    213703
1.0      4631
2.0     35346
Name: count, dtype: int64


In [7]:
# Check for Missing Values
print("Missing values:", df.isnull().sum().sum())

# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Display data types
print(df.dtypes)

Missing values: 0
Duplicate rows: 23899
Diabetes_012            float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object


In [12]:

# Binary features (0 or 1)
binary_cols = [
    "HighBP", "HighChol", "CholCheck",
    "Smoker", "Stroke", "HeartDiseaseorAttack",
    "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare",
    "NoDocbcCost", "DiffWalk", "Sex"
]

# Ordinal features (ordered categories)
ordinal_cols = [
    "GenHlth", "Age", "Education", "Income"
]

# Numerical features
numerical_cols = [
    "BMI", "MentHlth", "PhysHlth"
]


# Define our input features and target
X = df.drop(columns=["Diabetes_012"])
y = df["Diabetes_012"].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

# Check that all 21 features are accounted for
all_cols = binary_cols + ordinal_cols + numerical_cols

assert len(all_cols) == len(set(all_cols))
assert set(all_cols) == set(X.columns)

print("Binary features:", len(binary_cols))
print("Ordinal features:", len(ordinal_cols))
print("Numerical features:", len(numerical_cols))
print("Total features:", len(all_cols))



Features: (253680, 21)
Target: (253680,)
Binary features: 14
Ordinal features: 4
Numerical features: 3
Total features: 21


In [13]:

from sklearn.model_selection import train_test_split

# Split into 80% training and 20% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Divide the temporary data into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


Training: (202944, 21)
Validation: (25368, 21)
Testing: (25368, 21)


In [14]:

# Show percentage of each diabetes class in training
print(y_train.value_counts(normalize=True) * 100)

Diabetes_012
0    84.240973
2    13.933400
1     1.825627
Name: proportion, dtype: float64


In [15]:

# Check validation class percentages
print(y_val.value_counts(normalize=True) * 100)

# Check testing class percentages
print(y_test.value_counts(normalize=True) * 100)


Diabetes_012
0    84.239987
2    13.934879
1     1.825134
Name: proportion, dtype: float64
Diabetes_012
0    84.243929
2    13.930937
1     1.825134
Name: proportion, dtype: float64


In [16]:

# Identify matching feature combinations
train_keys = pd.util.hash_pandas_object(X_train, index=False)
test_keys = pd.util.hash_pandas_object(X_test, index=False)

# Count matching test records
overlap = test_keys.isin(set(train_keys)).sum()

print("Overlapping test records:", overlap)


Overlapping test records: 3480
